# 06. Feature Table Construction for ML Layer

**Stage:** 06_ml  
**Inputs:** `data/processed/clean_fixtures.parquet`, `data/processed/ratings.parquet`, `data/processed/adjusted_lambdas.parquet`, `data/processed/mc_probabilities.parquet`, `src/config/config.yaml`  
**Outputs:** `data/processed/feature_table.parquet`  

This notebook constructs the single-row-per-fixture feature table for XGBoost training and walk-forward validation (Task 4.2). It merges all upstream pipeline outputs, derives point-in-time non-leaking rolling form and xG differentials, rating differentials, rest days, head-to-head win rates, and fallback rating confidence indicators while preserving the 760-fixture dataset.

## 1. Environment & Setup

In [ ]:
# Load configuration and imports
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Locate repo root dynamically
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config.loader import load_config, get_form_windows, get_xg_windows
from src.ml.feature_table import construct_feature_table

config = load_config()
pts_window, goals_window = get_form_windows(config)
xg_for_win, xg_against_win = get_xg_windows(config)

processed_dir = repo_root / "data" / "processed"
clean_file = processed_dir / "clean_fixtures.parquet"
ratings_file = processed_dir / "ratings.parquet"
lambdas_file = processed_dir / "adjusted_lambdas.parquet"
mc_file = processed_dir / "mc_probabilities.parquet"
output_file = processed_dir / "feature_table.parquet"

print(f"Loaded config. Form windows: pts={pts_window}, goals={goals_window}. xG windows: for={xg_for_win}, against={xg_against_win}.")

## 2. Load & Audit Upstream Datasets

In [ ]:
assert clean_file.exists(), f"Missing input file: {clean_file}"
assert ratings_file.exists(), f"Missing input file: {ratings_file}"
assert lambdas_file.exists(), f"Missing input file: {lambdas_file}"
assert mc_file.exists(), f"Missing input file: {mc_file}"

df_clean = pd.read_parquet(clean_file)
df_ratings = pd.read_parquet(ratings_file)
df_lambdas = pd.read_parquet(lambdas_file)
df_mc = pd.read_parquet(mc_file)

print(f"Loaded clean fixtures: {len(df_clean)} rows")
print(f"Loaded ratings: {len(df_ratings)} rows")
print(f"Loaded adjusted lambdas: {len(df_lambdas)} rows")
print(f"Loaded MC probabilities: {len(df_mc)} rows")

# Re-run team-name reconciliation audit across all 4 upstream tables
teams_clean = set(df_clean["home_team"]).union(set(df_clean["away_team"]))
teams_ratings = set(df_ratings["home_team"]).union(set(df_ratings["away_team"]))
teams_lambdas = set(df_lambdas["home_team"]).union(set(df_lambdas["away_team"]))
teams_mc = set(df_mc["home_team"]).union(set(df_mc["away_team"]))

assert teams_clean == teams_ratings, f"Team mismatch clean vs ratings: {teams_clean ^ teams_ratings}"
assert teams_clean == teams_lambdas, f"Team mismatch clean vs lambdas: {teams_clean ^ teams_lambdas}"
assert teams_clean == teams_mc, f"Team mismatch clean vs mc: {teams_clean ^ teams_mc}"
print("Audit passed: 100% team-name consistency across all 4 upstream tables.")

## 3. Construct Feature Table

In [ ]:
df_feature_table = construct_feature_table(
    df_clean=df_clean,
    df_ratings=df_ratings,
    df_lambdas=df_lambdas,
    df_mc=df_mc,
    config=config
)

assert len(df_feature_table) == len(df_clean), f"Row count changed! Expected {len(df_clean)}, got {len(df_feature_table)}"
print(f"Successfully constructed feature table with {len(df_feature_table)} rows and {df_feature_table.shape[1]} columns.")
df_feature_table.head(3)

## 4. Feature EDA, Missing Values & Fallback Distribution Analysis

In [ ]:
# NBVAL_IGNORE_OUTPUT
print("=== Missing-Value Rate Per Column ===")
missing_series = (df_feature_table.isna().mean() * 100).round(2)
print(missing_series[missing_series > 0])

print("\n=== Fallback Rating Distribution Summary ===")
if "is_fallback_rating" in df_feature_table.columns:
    print(df_feature_table["is_fallback_rating"].value_counts())
else:
    print("is_fallback_rating column not found")

# Correlation Heatmap of Differential Features & MC Probabilities
numeric_features = [
    "points_form_diff",
    "goals_form_diff",
    "xg_for_form_diff",
    "xg_diff",
    "poisson_rating_diff",
    "adjusted_lambda_diff",
    "rest_days_diff",
    "h2h_home_win_rate",
    "mc_prob_home_win",
    "mc_prob_away_win",
    "target_encoded"
]

plt.figure(figsize=(10, 8))
sns.heatmap(df_feature_table[numeric_features].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Save Output Artifact

In [ ]:
# NBVAL_IGNORE_OUTPUT
output_file.parent.mkdir(parents=True, exist_ok=True)
df_feature_table.to_parquet(output_file, index=False)
print(f"Saved {len(df_feature_table)} rows to: {output_file}")